In [1]:
import hashlib
import os
import random

import numpy as np
import pandas as pd
import torch
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from lib.routed_sentence_transformer import RoutedSentenceTransformer
from sentence_transformers.sentence_transformer.model import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap.umap_ import UMAP
from underthesea import word_tokenize

df = pd.read_parquet("../data/VietJobs.parquet")

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48092 entries, 0 to 48091
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   job_title            48092 non-null  str  
 1   location             48092 non-null  str  
 2   country              48092 non-null  str  
 3   qualifications       47722 non-null  str  
 4   technical_skills     41655 non-null  str  
 5   soft_skills          45451 non-null  str  
 6   languages_required   12296 non-null  str  
 7   experience_required  48092 non-null  str  
 8   salary               48092 non-null  str  
 9   contract_type        48092 non-null  str  
 10  working_hours        43110 non-null  str  
 11  benefits             48053 non-null  str  
 12  description          48092 non-null  str  
 13  requirements_text    48083 non-null  str  
 14  category             48092 non-null  str  
 15  salary_min           48092 non-null  int64
 16  salary_max           48092 non-nu

In [3]:
df

,job_title,location,country,qualifications,technical_skills,soft_skills,languages_required,experience_required,salary,contract_type,working_hours,benefits,description,requirements_text,category,salary_min,salary_max,salary_avg
0,Nhân Viên Vận Hành Hệ Thống Xử Lý Nước Thải,hồ chí minh,Việt Nam,"['Cao đẳng', 'Đại học chuyên ngành môi trường'...",NaN,"['Sức khoẻ tốt', 'Nhiệt tình', 'Chịu khó', 'Tr...",NaN,1 năm,15.0 - 22.0 triệu,Toàn thời gian,"Thứ 2 - Thứ 6 (từ 08:00 đến 17:00), Thứ 7 làm ...","['Thưởng năm', 'Lương tháng 13', 'Bảo hiểm y t...",- Vận hành hằng ngày các công trình được giao ...,"- Tốt nghiệp cao đẳng, đại học chuyên ngành mô...",nông_nghiệp_năng_lượng_môi_trường,15,22,18 triệu
1,Nhân Viên Vận Hành Hệ Thống Xử Lí Nước Thải,hồ chí minh,Việt Nam,"['Tốt nghiệp Trung cấp trở lên', 'Chuyên ngành...","['Vận hành hệ thống xử lý nước thải', 'Phân tí...","['Chăm chỉ', 'Có tinh thần trách nhiệm cao']",NaN,Không yêu cầu,8.0 - 13.0 triệu,Toàn thời gian,Làm việc theo ca,"['Chế độ phụ cấp công trình', 'Hưởng đầy đủ cá...","Vận hành, giám sát hệ thống xử lý nước thải th...",Giới tính: Nam. Độ tuổi: tuân thủ luật lao độn...,nông_nghiệp_năng_lượng_môi_trường,8,13,10 triệu
2,Nhân Viên Phát Triển Thị Trường,vĩnh long,Việt Nam,['Cao đẳng trở lên các chuyên ngành về Nông ng...,NaN,"['Tự tin giao tiếp', 'Diễn đạt thông tin tốt',...",NaN,Không yêu cầu,9.4 - 12.0 triệu,Toàn thời gian,"Thứ 2 - Thứ 6 (từ 07:30 đến 17:00), Thứ 7 (từ ...","['Phụ cấp điện thoại', 'Phụ cấp xăng xe', 'Thư...",Thực hiện quy trình thử nghiệm và đánh giá hạt...,Tốt nghiệp Cao đẳng trở lên các chuyên ngành v...,nông_nghiệp_năng_lượng_môi_trường,9,12,10 triệu
3,Nhân Viên Bảo Trì Hệ Thống Xử Lý Nước,nghệ an,Việt Nam,"['Tốt nghiệp CĐ, ĐH chuyên ngành Môi trường', ...",NaN,"['Sức khỏe tốt', 'Lanh lợi', 'Hoạt bát', 'Chịu...",NaN,1 năm,12.0 - 15.0 triệu,Toàn thời gian,Thứ 2 - Thứ 6 (từ 08:00 đến 17:00),"['Phụ cấp: ăn trưa, xăng xe, điện thoại, công ...","Bảo trì bảo dưỡng các hệ thống xử lý nước, khí...","Trình độ: Tốt nghiệp CĐ, ĐH chuyên ngành Môi t...",nông_nghiệp_năng_lượng_môi_trường,12,15,13 triệu
4,Chuyên Gia Đánh Giá ISO 22000–HACCP–VIETGAP,"hà nội, hồ chí minh",Việt Nam,"['Đại học khối ngành kỹ thuật, môi trường']","['HACCP', 'ISO 22000', 'VietGAP', 'GLOBAL GAP']","['Khả năng đi công tác', 'Linh động trong việc...",NaN,2 năm,15.0 - 25.0 triệu,Toàn thời gian,"Thứ Hai- Thứ Sáu, từ 08h30-17h00, nghỉ trưa từ...","['Thưởng tháng 13', 'Thưởng Lễ, Tết', 'Du lịch...",Bạn sẽ làm gì tại SUTECH? Triển khai tư vấn hệ...,Ưu tiên Nam. Chuyên môn Tốt nghiệp Đại học khố...,nông_nghiệp_năng_lượng_môi_trường,15,25,20 triệu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48087,Trưởng Nhóm Chăm Sóc Khách Hàng,hà nội,Việt Nam,"['Tốt nghiệp Cao đẳng, Đại học các ngành marke...","['Sử dụng thành thạo CRM (VD: Hubspot, Salesfo...","['Giao tiếp và làm việc liên phòng ban', 'Làm ...",NaN,2 năm,20.0 - 30.0 triệu,Toàn thời gian,+Ca hành chính: 08h00 - 17h00\n+Ca chiều: từ 1...,['Chế độ Nghỉ và Thưởng vào các ngày Lễ lớn tr...,1. Quản lý hệ thống CRM và xây dựng chương trì...,"- Tốt nghiệp Cao đẳng, Đại học các ngành marke...",tài_chính_kế_toán_ngân_hàng_bảo_hiểm,20,30,25 triệu
48088,Nhân Viên Trung Tâm Nghiên Cứu Khoa Học,hà nội,Việt Nam,['Tốt nghiệp Đại học/sau Đại học hệ chính quy ...,['MS Office'],"['khả năng viết tốt', 'tư duy logic tốt', 'làm...",tiếng anh,1 năm,8.0 - 12.0 triệu,Toàn thời gian,Thứ 2 - Thứ 6 (từ 08:00 đến 17:30),"['Làm việc trong môi trường năng động, chuyên ...",* Trình Lãnh đạo Đơn vị nghiệp vụ tổ chức cá...,"Về ngoại hình: dễ nhìn, thân thiện và hòa đồng...",tài_chính_kế_toán_ngân_hàng_bảo_hiểm,8,12,10 triệu
48089,Nhân Viên Kế Toán,hồ chí minh,Việt Nam,['Tốt nghiệp Cao đẳng/Đại học chuyên ngành Kế ...,"['Kỹ năng tin học văn phòng', 'sử dụng tốt các...","['Kỹ năng giao tiếp tốt', 'chủ động trong công...",NaN,2 năm,12.0 - 15.0 triệu,Toàn thời gian,Thứ 2 - Thứ 6 (từ 08:00 đến 17:00)\nSáng thứ 7...,"['Được hưởng lương tháng 13', 'thưởng lễ, t

In [4]:
SEED = 42  # random seed for everything
TOPIC_COLUMNS = [
    "job_title",
    "description",
]  # columns to combine into a document
EMBEDDING_MODEL_NAME = "codefuse-ai/F2LLM-v2-1.7B"  # embedding model
ADD_COLUMN_TITLE = False  # add a title to each column's contents to seperate columns from each other in a document
BATCH_SIZE = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

hasher = hashlib.new("sha256")
hasher.update(bytes([ADD_COLUMN_TITLE]))
hasher.update("".join(TOPIC_COLUMNS).encode())
hasher.update(EMBEDDING_MODEL_NAME.encode())

parts = []
for col in TOPIC_COLUMNS:
    # Humanize the snake-case column name (e.g., 'product_title' -> 'Product Title')
    label = col.replace("_", " ").title()

    # Construct the vectorized string chunk for this column
    chunk = ((label + ": ") if ADD_COLUMN_TITLE else "") + df[col].fillna(
        "(missing)"
    ).astype(str).str.strip()
    parts.append(chunk)

# Join all chunks together vectorially using pandas addition (+)
# We use a separator like "\n\n" between each column expression
df["combined_text"] = parts[0]
for p in parts[1:]:
    df["combined_text"] = df["combined_text"] + "\n\n" + p

if not os.path.exists("../cache"):
    os.mkdir("../cache")

tokenization_path = f"../cache/tokenization_{hasher.hexdigest()}.npy"

docs = df["combined_text"].to_numpy()

if os.path.exists(tokenization_path):
    segmented_docs = np.load(tokenization_path)
else:
    tokenizer = np.vectorize(lambda doc: word_tokenize(doc, format="text"))
    segmented_docs = tokenizer(docs)
    np.save(tokenization_path, segmented_docs)

In [5]:
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cuda",
    model_kwargs={
        "attn_implementation": "sdpa",
        "torch_dtype": "bfloat16",
    },
)
embedding_model.compile(dynamic=True)

embedding_model = RoutedSentenceTransformer(embedding_model)

embeddings_path = f"../cache/embeddings_{hasher.hexdigest()}.npy"

if os.path.exists(embeddings_path):
    embeddings = np.load(embeddings_path)
else:
    embeddings = embedding_model.encode(
        docs,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(embeddings_path, embeddings)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/3006 [00:00<?, ?it/s]

In [6]:
vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    stop_words=["vietnamese", "english"],
    ngram_range=(1, 2),
    max_df=0.9,
)
umap_model = UMAP(random_state=SEED)
hdbscan_model = HDBSCAN()
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
representation_model = [
    KeyBERTInspired(top_n_words=30),
    MaximalMarginalRelevance(diversity=0.4),
]

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    ctfidf_model=ctfidf_model,
    hdbscan_model=hdbscan_model,
    representation_model=representation_model,
    zeroshot_min_similarity=0.5,
)
with embedding_model.query_mode():
    topics, probs = topic_model.fit_transform(
        segmented_docs,
        embeddings=embeddings,
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
topic_model.get_topic_info().to_csv("../sus.csv")